# ML - Ephyrae Appearance Prediction

**Goal:** Predict ephyrae appearance for well-documented species (Aurelia labiata) using polyp count, temperature, and context events (repiquage, temperature changes).

**Date:** 2026-05-31

## 1. Setup and data loading

In [ ]:
from pathlib import Path
import re
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = Path('../data')

TRACKING_CSV = DATA_DIR / 'polyp_tracking_long_format.csv'
CHANGES_FILE = DATA_DIR / 'ChangementT°C_Repiquage.xlsx'

print('Checking data files...')
print(f'Tracking CSV exists: {TRACKING_CSV.exists()}')
print(f'Changes file exists: {CHANGES_FILE.exists()}')

## 2. Load tracking data

In [ ]:
# Load tracking data with error handling
if not TRACKING_CSV.exists():
    print(f'ERROR: {TRACKING_CSV} not found!')
    print('Please run excel_eda.ipynb first to generate the CSV.')
    tracking_df = pd.DataFrame()
else:
    try:
        tracking_df = pd.read_csv(TRACKING_CSV, encoding='utf-8-sig')
        print(f'✓ Loaded {len(tracking_df)} rows')
        print(f'✓ Columns: {tracking_df.columns.tolist()}')
        
        species_counts = (
            tracking_df[tracking_df['value'].notna()]
            .groupby('species')
            .agg({
                'value': 'count',
                'box': 'nunique',
                'file_year': 'nunique'
            })
            .rename(columns={'value': 'measurements', 'box': 'boxes', 'file_year': 'years'})
            .sort_values('measurements', ascending=False)
        )
        
        print('\nTop 10 species by measurement count:')
        display(species_counts.head(10))
        
        TARGET_SPECIES = 'Aurelia labiata'
        print(f'\n✓ Selected species: {TARGET_SPECIES}')
    except Exception as e:
        print(f'ERROR loading CSV: {e}')
        tracking_df = pd.DataFrame()

## 3. Load repiquage/temperature change events

In [ ]:
def load_events_from_excel(file_path, target_species):
    """Load repiquage and temperature change events."""
    events = {}
    try:
        workbook = pd.ExcelFile(file_path)
        for sheet in workbook.sheet_names:
            year_match = re.search(r'(20\d{2})', sheet)
            if not year_match:
                continue
            year = int(year_match.group(1))
            raw = pd.read_excel(file_path, sheet_name=sheet, header=None)
            
            for idx, row in raw.iterrows():
                first_cell = str(row.iloc[0]).strip() if pd.notna(row.iloc[0]) else ''
                if target_species.lower() in first_cell.lower():
                    for month_idx in range(2, min(14, len(row))):
                        if pd.notna(row.iloc[month_idx]):
                            month = month_idx - 1
                            event_key = (target_species, month, year)
                            events[event_key] = 1
    except Exception as e:
        print(f'Warning loading events: {e}')
    return events

if not tracking_df.empty:
    events = load_events_from_excel(CHANGES_FILE, TARGET_SPECIES)
    print(f'✓ Loaded {len(events)} event markers for {TARGET_SPECIES}')
else:
    events = {}
    print('Skipping event loading (no tracking data)')

## 4. Prepare data for target species

In [ ]:
if not tracking_df.empty:
    species_data = tracking_df[tracking_df['species'] == TARGET_SPECIES].copy()
    
    print(f'Data points for {TARGET_SPECIES}: {len(species_data)}')
    print(f'Boxes: {species_data["box"].nunique()}')
    print(f'Years: {species_data["file_year"].nunique()}')
    
    pivot_data = species_data.pivot_table(
        index=['box', 'file_year', 'week_index', 'temperature', 'group'],
        columns='normalized_measurement_type',
        values='value',
        aggfunc='first'
    ).reset_index()
    pivot_data['species'] = TARGET_SPECIES
    
    print(f'\nPivoted shape: {pivot_data.shape}')
    display(pivot_data.head(10))
    
    print(f'\nEphyrae statistics:')
    print(pivot_data['ephyrae'].describe())
    print(f'Non-zero ephyrae: {(pivot_data["ephyrae"] > 0).sum()}')
    print(f'Zero ephyrae: {(pivot_data["ephyrae"] == 0).sum()}')
    print(f'Missing ephyrae: {pivot_data["ephyrae"].isna().sum()}')
else:
    pivot_data = pd.DataFrame()
    print('No data to process')

## 5. Create features for ML model

In [ ]:
if not pivot_data.empty:
    def create_features(df, events_dict):
        """Create features for ML: polyp trend, temperature, seasonality, events."""
        ml_df = df.copy()
        ml_df = ml_df.sort_values(['box', 'file_year', 'week_index'])
        
        # Target: ephyrae present (1) or not (0)
        ml_df['ephyrae_target'] = (ml_df['ephyrae'] > 0).astype(int)
        ml_df['polyp_count'] = ml_df['polyps']
        ml_df['week_of_year'] = ml_df['week_index'] % 52
        
        # Rolling stats by box
        ml_df['polyps_rolling_mean_4w'] = (
            ml_df.groupby('box')['polyp_count']
            .transform(lambda x: x.rolling(window=4, min_periods=1).mean())
        )
        
        ml_df['polyps_rolling_std_4w'] = (
            ml_df.groupby('box')['polyp_count']
            .transform(lambda x: x.rolling(window=4, min_periods=1).std())
        )
        ml_df['polyps_rolling_std_4w'].fillna(0, inplace=True)
        
        # Event indicator
        ml_df['has_recent_event'] = (
            ml_df.apply(
                lambda row: events_dict.get((row['species'], row['week_of_year'], int(row['file_year'])), 0),
                axis=1
            )
        )
        return ml_df
    
    ml_ready = create_features(pivot_data, events)
    
    print('✓ Features created:')
    display(ml_ready[[
        'box', 'file_year', 'week_index', 'polyp_count', 'temperature',
        'ephyrae', 'ephyrae_target', 'week_of_year', 'polyps_rolling_mean_4w',
        'polyps_rolling_std_4w', 'has_recent_event'
    ]].head(15))
    
    print(f'\nTarget distribution:')
    print(ml_ready['ephyrae_target'].value_counts(dropna=False))
else:
    ml_ready = pd.DataFrame()
    print('No features to create')

## 6. Prepare training data

In [ ]:
if not ml_ready.empty:
    train_data = ml_ready.dropna(subset=[
        'polyp_count', 'temperature', 'ephyrae_target',
        'polyps_rolling_mean_4w', 'polyps_rolling_std_4w'
    ]).copy()
    
    print(f'✓ Training rows available: {len(train_data)}')
    print(f'\nTarget balance:')
    print(train_data['ephyrae_target'].value_counts())
    print(f'Percentage with ephyrae: {train_data["ephyrae_target"].mean()*100:.1f}%')
    
    FEATURE_COLS = [
        'polyp_count',
        'temperature',
        'week_of_year',
        'polyps_rolling_mean_4w',
        'polyps_rolling_std_4w',
        'has_recent_event'
    ]
    
    X = train_data[FEATURE_COLS].copy()
    y = train_data['ephyrae_target'].copy()
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = pd.DataFrame(X_scaled, columns=FEATURE_COLS, index=X.index)
    
    print(f'\nFeature correlation with target:')
    corr_with_target = pd.concat([X_scaled, y], axis=1).corr()['ephyrae_target'].drop('ephyrae_target').sort_values(ascending=False)
    display(corr_with_target)
else:
    train_data = pd.DataFrame()
    print('No training data available')

## 7. Train classification model

In [ ]:
if not train_data.empty:
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f'✓ Training set: {len(X_train)} samples')
    print(f'✓ Test set: {len(X_test)} samples')
    print(f'Train target balance: {y_train.mean():.1%} with ephyrae')
    print(f'Test target balance: {y_test.mean():.1%} with ephyrae')
    
    print('\nTraining GradientBoostingClassifier...')
    model = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
        verbose=0
    )
    
    model.fit(X_train, y_train)
    print('✓ Model trained!')
    
    cv_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='roc_auc')
    print(f'\n✓ Cross-validation ROC-AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})')
else:
    print('No model to train')

## 8. Evaluate model

In [ ]:
if not train_data.empty:
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    y_pred_proba_test = model.predict_proba(X_test)[:, 1]
    
    train_accuracy = (y_pred_train == y_train).mean()
    test_accuracy = (y_pred_test == y_test).mean()
    test_auc = roc_auc_score(y_test, y_pred_proba_test)
    
    print(f'Train Accuracy: {train_accuracy:.3f}')
    print(f'Test Accuracy: {test_accuracy:.3f}')
    print(f'Test ROC-AUC: {test_auc:.3f}')
    
    print(f'\nClassification Report (Test Set):')
    print(classification_report(y_test, y_pred_test, target_names=['No Ephyrae', 'Ephyrae']))
    
    cm = confusion_matrix(y_test, y_pred_test)
    print(f'\nConfusion Matrix:')
    print(cm)

## 9. Feature importance

In [ ]:
if not train_data.empty:
    feature_importance = pd.DataFrame({
        'feature': FEATURE_COLS,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print('Feature Importance:')
    display(feature_importance)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    feature_importance.plot(kind='barh', x='feature', y='importance', ax=ax, legend=False)
    ax.set_xlabel('Importance')
    ax.set_title(f'Feature Importance - {TARGET_SPECIES}')
    plt.tight_layout()
    plt.show()

## 10. ROC Curve

In [ ]:
if not train_data.empty:
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_test)
    
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(fpr, tpr, label=f'ROC Curve (AUC={test_auc:.3f})', lw=2)
    ax.plot([0, 1], [0, 1], 'k--', label='Random')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve - {TARGET_SPECIES}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 11. Predictions on recent data (2026)

In [ ]:
if not train_data.empty and not ml_ready.empty:
    recent_data = ml_ready[
        (ml_ready['file_year'] == 2026) &
        (ml_ready[FEATURE_COLS].notna().all(axis=1))
    ].copy()
    
    if len(recent_data) > 0:
        X_recent = recent_data[FEATURE_COLS].copy()
        X_recent_scaled = scaler.transform(X_recent)
        
        recent_proba = model.predict_proba(X_recent_scaled)[:, 1]
        recent_pred = model.predict(X_recent_scaled)
        
        recent_data['ephyrae_probability'] = recent_proba
        recent_data['ephyrae_predicted'] = recent_pred
        
        high_risk = recent_data[recent_data['ephyrae_probability'] > 0.5].sort_values(
            'ephyrae_probability', ascending=False
        )[['box', 'week_index', 'polyp_count', 'temperature', 'ephyrae', 'ephyrae_probability']]
        
        print(f'🚨 High-risk observations (probability > 0.5) in 2026:')
        display(high_risk.head(20))
        print(f'\nTotal 2026 predictions: {len(recent_data)}')
        print(f'High-risk count: {len(high_risk)}')
    else:
        print('No complete 2026 data available yet')
else:
    print('Cannot make predictions (no trained model)')

## 12. Summary

**Model Performance:** Trained GradientBoosting classifier for ephyrae prediction on Aurelia labiata

**Key Insights:**
- Model identifies boxes at high risk of ephyrae production
- Temperature, polyp count, and seasonality are key predictors
- Actionable alerts can be generated for facility staff

**Next Steps:**
1. Apply same pipeline to other well-documented species
2. Fine-tune event parsing from Excel files
3. Add more lagged features (7-day, 14-day trends)
4. Deploy as real-time scoring API
5. Integrate into webapp with threshold alerts